# Spark on the cluster from a local notebook (Livy + sparkmagic)

This notebook runs on **your laptop** with a plain Python 3 kernel. `sparkmagic` opens a **Livy session** on the cluster
and sends the code of every `%%spark` cell to it as a *statement* — the same REST calls you made with `curl` in the lab.
The Spark code runs in a YARN container, as **you**; your laptop only receives small results.

Run the cells from top to bottom. Close the session in the last cell, so it does not keep cluster resources.

## How to read this notebook: where does a cell run?

| Cell starts with | Runs on | Typical use |
|---|---|---|
| nothing (plain Python) | **your laptop** | ask for the password, plot a small pandas DataFrame |
| `%spark ...` (one `%`) | **your laptop**, talks to Livy | manage sessions: `add`, `info`, `logs`, `delete` |
| `%%spark -s workshop` (two `%%`) | **the cluster** (the Livy session) | PySpark code; one cell = one Livy *statement* |
| `%%spark -s workshop -c sql` | **the cluster** | a Spark SQL query; the result comes back as a table |
| `%send_to_spark ...` | **your laptop → the cluster** | copy a local variable into the session (section 4) |

**One session, many cells.** All `%%spark` cells with the same `-s` name share one Spark driver. A variable, DataFrame, temp view or
import from an earlier cell is still there in the next one. So you do **not** need one big cell: write one cell per step, and run
them one after another. A failing cell does not end the session; fix the cell and run it again.

### What the flags mean

The same letter means different things in different magics. This is the full list of what this notebook uses.

**`%spark add` — create a session** (the `POST /sessions` of the lab)

| Flag | Meaning |
|---|---|
| `-s workshop` | **s**ession name. A label that only exists in this notebook; later cells refer to it |
| `-l python` | **l**anguage of the session: `python` (PySpark), `scala`, or `r` |
| `-u URL` | Livy **u**RL (the Knox address from your handout) |
| `-a USER`, `-p PASSWORD` | user name and password for the LDAP login |
| `-t Basic_Access` | authentication **t**ype: `Basic_Access` = user + password, `Kerberos`, or `None` |
| `-k` | **k**eep going: do nothing if a session with this name already exists (safe to re-run the cell) |

**`%%spark` — run a cell on the cluster**

| Flag | Meaning |
|---|---|
| `-s workshop` | which session runs the cell (optional if you have only one session) |
| `-c sql` | **c**ontext: `spark` (default, PySpark code) or `sql` (the cell is a SQL query) |
| `-o name` | **o**utput: copy the result into a **pandas** DataFrame called `name` on your laptop. For `-c spark` the cell must define a DataFrame with that name |
| `-m take` | sampling **m**ethod when copying: `take` (first rows) or `sample` (random rows) |
| `-n 100` | **n**umber: at most 100 rows are copied to the laptop (default 2500) |
| `-r 0.1` | sample **r**atio for `-m sample` (10 % of the rows) |
| `-q` | **q**uiet: no table or chart for a SQL cell (useful together with `-o`) |
| `-e false` | **e**: type coercion off. By default sparkmagic converts text columns that look like dates into pandas datetimes when it copies a result; `false` keeps them as text |

**`%send_to_spark` — copy a laptop variable to the cluster**

| Flag | Meaning |
|---|---|
| `-i workshop_root` | **i**nput: the name of the variable on your laptop |
| `-t str` | **t**ype of the variable: `str` (a string) or `df` (a pandas DataFrame) |
| `-n workshop_root` | **n**ame the variable gets on the cluster (default: the same as `-i`) |
| `-m 2500` | **m**ax rows, only for `df` |
| `-s workshop` | session that receives the variable (default `workshop`) |

> `send_to_spark` is not part of `%spark`; sparkmagic only ships it inside its own *PySpark kernel*, where it is written `%%send_to_spark`.
> In this plain Python notebook section 2 defines it for you with the same flags. Because IPython does not allow a cell magic with an empty
> body, here it is a line magic with **one** `%`.

## 1. Connection data

The URL comes from your handout (no trailing slash). The password is asked with `getpass`: it is kept in memory
of this kernel only and never written into the notebook file.

In [ ]:
import getpass

livy_url = input("Livy URL (from your handout, no trailing slash): ").strip().rstrip("/")
ws_user = input("User name (WORKSHOP_USER): ").strip()
ldap_password = getpass.getpass(f"LDAP password for {ws_user}: ")
workshop_root = input("WORKSHOP_ROOT [/2026_cdp_workshop]: ").strip() or "/2026_cdp_workshop"
workshop_home = f"/user/{ws_user}"

## 2. Load `sparkmagic`, add `%send_to_spark`, open a session

`%load_ext` registers the `%spark` and `%%spark` magics.

In [ ]:
%load_ext sparkmagic.magics

The next cell defines `%send_to_spark` with the flags `-i`, `-t`, `-n`, `-m` and `-s` described above. It reuses the Livy
connection of `%spark` and sparkmagic's own classes that turn a string or a DataFrame into PySpark code. Run it once; there is nothing to change.

In [ ]:
import argparse, shlex
from IPython import get_ipython
from sparkmagic.livyclientlib.sendstringtosparkcommand import SendStringToSparkCommand
from sparkmagic.livyclientlib.sendpandasdftosparkcommand import SendPandasDfToSparkCommand

def send_to_spark(line):
    """Line magic: %send_to_spark -i VAR [-t str|df] [-n NAME] [-m MAXROWS] [-s SESSION]"""
    p = argparse.ArgumentParser(prog="%send_to_spark")
    p.add_argument("-i", "--input", required=True)
    p.add_argument("-t", "--vartype", default="str", choices=["str", "df"])
    p.add_argument("-n", "--varname")
    p.add_argument("-m", "--maxrows", type=int, default=2500)
    p.add_argument("-s", "--session", default="workshop")
    a = p.parse_args(shlex.split(line))
    ip = get_ipython()
    value = ip.user_ns[a.input]
    name = a.varname or a.input
    command = (SendStringToSparkCommand(a.input, value, name) if a.vartype == "str"
               else SendPandasDfToSparkCommand(a.input, value, name, a.maxrows))
    controller = ip.magics_manager.registry["RemoteSparkMagics"].spark_controller
    ok, result, _ = controller.run_command(command, a.session)
    print(f"Sent '{a.input}' as '{name}' to session '{a.session}'" if ok else f"FAILED: {result}")

get_ipython().register_magic_function(send_to_spark, "line", "send_to_spark")

`%spark add` is the `POST /sessions` of the lab. Starting the session takes about a minute; a progress bar shows `starting` and then `idle`.

In [ ]:
%spark add -s workshop -l python -u $livy_url -a $ws_user -p $ldap_password -t Basic_Access -k

`%spark info` lists your session: the **Livy session id**, the **YARN application id**, and links to the Spark UI and the driver log.
Compare the application id with `yarn application -list` on the edge node.

In [ ]:
%spark info

## 3. Code on the cluster

Everything in a `%%spark` cell is Python **on the cluster**. `spark` is the `SparkSession` of the session.
Check where the cell really runs and which user it runs as.

In [ ]:
%%spark -s workshop
import getpass, socket
print("host :", socket.gethostname())
print("user :", getpass.getuser())
print("Spark:", spark.version)

## 4. Send local variables to the cluster

The variables `workshop_root` and `workshop_home` exist only in this notebook, not in the Spark driver. First prove it:

In [ ]:
%%spark -s workshop
try:
    print(workshop_root)
except NameError as e:
    print("NameError:", e)

The cell above catches the `NameError` and prints it: the cluster does not know the variable. Now copy the variables over under the same name (`-i` = laptop variable, `-t` = type, `-n` = name on the cluster).
The magic has one line and no cell body.

In [ ]:
%send_to_spark -i workshop_root -t str -n workshop_root

In [ ]:
%send_to_spark -i workshop_home -t str -n workshop_home

The same cell as before now works, because the variable lives in the Spark driver:

In [ ]:
%%spark -s workshop
print(workshop_root, workshop_home)

## 5. The pipeline from the lab, step by step

Each step is its own `%%spark` cell. They share one session, so each cell can use what the earlier ones defined.

**5.1 Imports and the meter data.** Read the meter CSV with an explicit schema.

In [ ]:
%%spark -s workshop
from pyspark.sql import functions as F

meter = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, station_id STRING, device_id STRING, phase STRING, "
        "active_power_kw DOUBLE, reactive_power_kvar DOUBLE, voltage_v DOUBLE, "
        "current_a DOUBLE, power_factor DOUBLE, frequency_hz DOUBLE, "
        "transformer_load_pct DOUBLE, quality_flag STRING, event_code STRING, "
        "comm_status STRING")
    .csv(workshop_root + "/data/grid_measurements/sample-grid-measurements.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
print("meter rows:", meter.count())

**5.2 The weather data.** Same idea, a second DataFrame in the same session.

In [ ]:
%%spark -s workshop
weather = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, weather_station_id STRING, station_name STRING, "
        "temperature_c DOUBLE, relative_humidity_pct INT, wind_speed_mps DOUBLE, "
        "precipitation_mm DOUBLE, pressure_hpa DOUBLE, weather_quality STRING")
    .csv(workshop_root + "/data/operator_weather/sample-operator-weather.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
print("weather rows:", weather.count())

Expect **15** meter rows and **10** weather rows. A quick look at the data, still on the cluster:

In [ ]:
%%spark -s workshop
meter.show(5, truncate=False)

**5.3 Cleanse.** Keep `OK` readings and drop duplicates. Expect **12** rows.

In [ ]:
%%spark -s workshop
meter_ok = (meter.filter(F.col("quality_flag") == "OK")
    .dropDuplicates(["ts_utc", "station_id", "device_id", "phase"]))
print("cleansed rows:", meter_ok.count())

Publish the result as a temporary view, so the next section can use plain SQL.

In [ ]:
%%spark -s workshop
meter_ok.createOrReplaceTempView("meter_ok")

## 6. A SQL cell

`-c sql` turns the cell into a Spark SQL query. The result comes back as a table.

In [ ]:
%%spark -s workshop -c sql
SELECT station_id, COUNT(*) AS readings, ROUND(AVG(active_power_kw), 2) AS avg_kw, MAX(transformer_load_pct) AS max_load_pct
FROM meter_ok
GROUP BY station_id
ORDER BY station_id

## 7. Bring a small result to your laptop

First build the DataFrame `hourly` on the cluster (no data travels yet).

In [ ]:
%%spark -s workshop
hourly = (meter_ok.withColumn("hour_utc", F.date_format(F.date_trunc("hour", F.col("ts_utc")), "yyyy-MM-dd HH:mm"))
    .groupBy("station_id", "hour_utc")
    .agg(F.avg("active_power_kw").alias("avg_power_kw"), F.max("transformer_load_pct").alias("max_load_pct"))
    .orderBy("station_id", "hour_utc"))

Now copy it. `-o hourly` creates a **pandas** DataFrame with the same name on your laptop, `-m take -n 100` limits it to the first 100 rows.
The cell body only has to *evaluate to* the DataFrame, so it is just its name. Only do this for **small** results — a large table would
travel through Knox and Livy as JSON.

In [ ]:
%%spark -s workshop -o hourly -m take -n 100
hourly

This cell runs **locally** on pandas and matplotlib — no cluster involved. Expect two hour buckets per station.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

print(type(hourly))
# sparkmagic turns text columns that look like dates into pandas datetimes, so format the column instead of slicing a string
hourly["label"] = hourly["station_id"] + " " + pd.to_datetime(hourly["hour_utc"]).dt.strftime("%H:%M")
hourly.plot.bar(x="label", y="avg_power_kw", legend=False, figsize=(9, 4), title="Average active power per station and hour")
plt.ylabel("kW")
plt.show()

## 8. Join, write, and verify

Join weather on the timestamp (two stations per timestamp, so **24** rows).

In [ ]:
%%spark -s workshop
curated = meter_ok.join(weather, ["ts_utc"], "inner")
print("curated rows:", curated.count())

Write Parquet to your HDFS home. Afterwards check the folder on the edge node: `hdfs dfs -ls /user/<your_user>/livy-nb-curated`.

In [ ]:
%%spark -s workshop
curated.write.mode("overwrite").partitionBy("station_id").parquet(workshop_home + "/livy-nb-curated")

## 9. Close the session

Closing the notebook does **not** stop the Livy session — the Spark application keeps running on YARN until Livy's idle timeout.
`%spark delete` is the `DELETE /sessions/<id>` of the lab. Run it every time.

In [ ]:
%spark delete -s workshop

Optional: `%spark cleanup` deletes **all** sessions that this kernel created, in case you opened more than one.

**Question:** Which cells were executed on your laptop, and which on the cluster? What happened in section 4 before you ran `%send_to_spark`,
and why?